# imports 

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

Convert the data to a data frame 

In [8]:
df = pd.read_csv("customer_churn_practice.csv")


# 1. Display the first five rows and dataset shape

In [9]:
df.head()

,customer_id,age,plan_type,months_active,monthly_fee,avg_weekly_usage_hours,support_tickets_90d,late_payments_12m,auto_pay,satisfaction_score,region,churned
0,CUST-0001,38,Basic,6,27.90,3.6,2,0,Yes,9.0,West,0
1,CUST-0002,32,Basic,14,25.89,7.8,3,1,Yes,4.0,South,1
2,CUST-0003,53,Basic,54,21.06,9.9,1,0,No,7.0,Midwest,0
3,CUST-0004,35,Standard,1,47.46,4.3,0,0,No,NaN,South,1
4,CUST-0005,47,Basic,33,29.96,7.7,3,0,Yes,6.0,West,0


# 2. Create X containing the input features
Remove `customer_id` and `churned` from X

In [10]:
X = df.drop(columns=["churned", "customer_id"])

I originally wrote X = df.drop("churned", "customer_id"). Pandas requires multiple column names to be placed inside a list. I also needed to use columns= to tell pandas that I wanted to remove columns rather than rows. The correct format is df.drop(columns=["churned", "customer_id"]).


# 3. Create y containing the target column

In [11]:
y = df["churned"]
print(y.head())
print(y.shape)


0    0
1    1
2    0
3    1
4    0
Name: churned, dtype: int64
(800,)


# 4. Display the shapes and columns of X and y

In [12]:
print(X.shape)
print(y.shape)

(800, 10)
(800,)


# 5. Split the data into training and testing sets
Use 20% for testing, `random_state=42`, and stratify the target

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y) # add stratify=y to ensure the class distribution is maintained in both train and test sets

# 6. Display the shapes of all four training and testing variables

In [14]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(640, 10) (160, 10) (640,) (160,)


# 7. Compare the churn percentages in `y_train` and `y_test`

In [18]:
percentTrain = y_train.value_counts(normalize=True) *100 # I tried to devide byu len(y_train) but i had normalize=True so it was already giving me the percentage after i multiplied by 100 to get the percentage
percentTest = y_test.value_counts(normalize=True) *100
print("Percent train y:", percentTrain)
print("Percent test y:", percentTest)


Percent train y: churned
0    68.4375
1    31.5625
Name: proportion, dtype: float64
Percent test y: churned
0    68.125
1    31.875
Name: proportion, dtype: float64


Stratification worked because the training and testing sets have almost the same class percentages. In the churned target, 0 represents a customer who did not churn, while 1 represents a customer who churned.

# 8. Create a list of numerical feature names

In [22]:
numerical_features = [
    "age", "months_active", "monthly_fee", "avg_weekly_usage_hours", "support_tickets_90d", "late_payments_12m", "satisfaction_score"
]

X[numerical_features].head()


,age,months_active,monthly_fee,avg_weekly_usage_hours,support_tickets_90d,late_payments_12m,satisfaction_score
0,38,6,27.90,3.6,2,0,9.0
1,32,14,25.89,7.8,3,1,4.0
2,53,54,21.06,9.9,1,0,7.0
3,35,1,47.46,4.3,0,0,NaN
4,47,33,29.96,7.7,3,0,6.0


# 9. Create a list of categorical feature names

In [23]:
categorical_features = ["plan_type", "auto_pay", "region"]

X[categorical_features].head()

,plan_type,auto_pay,region
0,Basic,Yes,West
1,Basic,Yes,South
2,Basic,No,Midwest
3,Standard,No,South
4,Basic,Yes,West


# 10. Create the numerical preprocessing steps
Replace missing values with the median
Scale the numerical features

In [26]:
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

What this means:
- Pipeline(...) runs steps in order.
- "imputer" fills missing numeric values.
- SimpleImputer(strategy="median") replaces missing values with the median of that column.
- "scaler" scales numeric values.
- StandardScaler() puts numeric columns on a similar scale, which helps Logistic Regression.

# 11. Create the categorical preprocessing steps
Replace missing values with the most frequent value
One-hot encode the categorical features

In [27]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

What this means:
- Missing category values get replaced with the most common category.
- OneHotEncoder turns text categories into numeric 0/1 columns.
- handle_unknown="ignore" prevents errors if the test set has a category that was not seen during training.

# 12. Combine the numerical and categorical preprocessing

In [28]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

What this means:
- ColumnTransformer applies different preprocessing to different columns.
- "num" applies numerical_transformer to your numerical_features.
- "cat" applies categorical_transformer to your categorical_features.
- This lets one pipeline handle both numeric and categorical data correctly.

# 13. Create and train a baseline DummyClassifier

In [29]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

baseline_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


What this means:
- This creates a baseline model.
- It first preprocesses the data.
- Then DummyClassifier(strategy="most_frequent") always predicts the most common class.
- In your dataset, that likely means it mostly predicts “stayed.”
- .fit(X_train, y_train) trains the pipeline on the training data.

# 14. Make baseline predictions and evaluate the baseline

In [30]:
baseline_predictions = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print("Precision:", precision_score(y_test, baseline_predictions))
print("Recall:", recall_score(y_test, baseline_predictions))
print("F1:", f1_score(y_test, baseline_predictions))

Accuracy: 0.68125
Precision: 0.0
Recall: 0.0
F1: 0.0


c:\Users\flemi\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


What this means:
- .predict(X_test) makes predictions on data the model did not train on.
- y_test contains the real answers.
- baseline_predictions contains the model’s guesses.
- The metric functions compare real answers against predicted answers.

# 15. Create a pipeline containing preprocessing and Logistic Regression

In [31]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

What this means:
- This creates your real Logistic Regression model pipeline.
- It uses the same preprocessing from #12.
- Then it trains Logistic Regression on the cleaned/encoded/scaled data.
- max_iter=1000 gives the model more time to converge.

# 16. Train the Logistic Regression pipeline

# 17. Make predictions using the testing data

# 18. Calculate accuracy, precision, recall, and F1 score

# 19. Display the classification report

# 20. Create and display the confusion matrix

# 21. Create a heatmap of the confusion matrix